# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring - which one, and why?*

**Ranking, built on a binary classifier.** The reviewer's question is "which pages first?", which is a ranking problem. A classifier supplies the score: the probability that a page's impressions will persistently fall in the next 30 days. The scores sort the review queue.

In [1]:
TASK = 'ranking via binary classification'
print('Task type:', TASK)

Task type: ranking via binary classification


## 2. Target or proxy

*What would you predict? Where does that label come from - observed outcome or a defined rule?*

**Target: `decline_next_30d`.** It equals 1 if, in the 30 days after snapshot t, the page's impressions fall at least 20% below its last 30 days relative to its own client's overall change over the same windows, and the drop persists (both 15-day halves are down). Otherwise 0. Pages with fewer than 100 impressions in the 30 days before t are not labeled.

This is an outcome observed in a later time window, not a product flag and not a rule on the features. The 20% cut and the persistence rule are provisional and are fixed in the Methodology label step before any training. Product flags (`health_score`, `priority_score`, `action_type`) are not in the data and are never used as features.

In [2]:
TARGET = 'decline_next_30d'
LABEL_RULE = {
    'min_prior_impressions_30d': 100,
    'relative_drop': 0.20,          # provisional, fixed at the label step
    'horizon_days': 30,
    'persistence': 'both 15-day halves down',  # provisional
}
print(TARGET, LABEL_RULE)

decline_next_30d {'min_prior_impressions_30d': 100, 'relative_drop': 0.2, 'horizon_days': 30, 'persistence': 'both 15-day halves down'}


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary: precision@20 and precision@50** on the sealed test month, because a reviewer checks 20 to 50 pages per cycle. Also reported: average precision and recall@K, with the base rate next to every number. The model must beat a transparent rule baseline on the same split; a tie or a loss is reported as it is. These metrics are named here, before any training.

In [3]:
K_LIST = [20, 50]
METRICS = ['precision@K', 'average precision', 'recall@K', 'base rate']
print('K values:', K_LIST)
print('Metrics:', METRICS)

K values: [20, 50]
Metrics: ['precision@K', 'average precision', 'recall@K', 'base rate']


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item at one month-end snapshot t**, written `(content_hash_id, t)`. The frame below is the starter sample: one row per page for a single window. The capstone repeats this shape for every snapshot t, built from the full warehouse.

In [4]:
import pandas as pd

STARTER_LOCAL = '../../data/raw/content_refresh_anonymized.csv'
STARTER_URL = 'https://raw.githubusercontent.com/maahilkureshi-design/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv'
try:
    df = pd.read_csv(STARTER_LOCAL)
except FileNotFoundError:
    df = pd.read_csv(STARTER_URL)

cols = ['impressions_prev_30d', 'impressions_last_30d', 'ctr', 'avg_position', 'trend_direction']
df[cols].head()

,impressions_prev_30d,impressions_last_30d,ctr,avg_position,trend_direction
0,987,578,0.76,10.6,down
1,5915,2501,0.05,20.3,down
2,6089,2382,0.09,36.5,down
3,4206,3626,0.49,6.2,stable
4,6452,4211,0.13,44.0,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule flags pages with one or two thresholds, for example "trend is down". Real decline depends on several signals at once: volume, position, CTR, page age, content type, and recent trend. A flag also cannot order pages inside the flagged set, and that set is far larger than a reviewer can work. The open question, answered in Results, is whether a model learned from past-to-future outcomes ranks better than a transparent rule on the same held-out month.

In [5]:
import pandas as pd

STARTER_LOCAL = '../../data/raw/content_refresh_anonymized.csv'
STARTER_URL = 'https://raw.githubusercontent.com/maahilkureshi-design/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv'
try:
    df = pd.read_csv(STARTER_LOCAL)
except FileNotFoundError:
    df = pd.read_csv(STARTER_URL)

capacity = 50
flagged = int((df['trend_direction'] == 'down').sum())
print(f"A plain 'down' flag marks {flagged:,} of {len(df):,} pages; a reviewer can act on about {capacity} per cycle.")

A plain 'down' flag marks 16,262 of 30,000 pages; a reviewer can act on about 50 per cycle.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.